In [ ]:
import ipywidgets as widgets
from IPython.display import display
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io

In [ ]:
from IPython.display import display, clear_output
import ipywidgets as widgets
from PIL import Image
import io

# 1. Initialize widgets
uploader = widgets.FileUpload(accept='image/*', multiple=False)
out = widgets.Output()  # This is the 'container' for your image

image_grey = None  # To store the result globally

def on_upload_change(change):
    global image_grey
    
    # We use 'with out' so display() knows exactly where to put the image
    with out:
        clear_output()  # Removes the previous image if you upload a new one
        
        # Check if a file was actually uploaded (handles the tuple in v8+)
        if not change['new']:
            return
            
        # Get the first file from the upload tuple
        file_info = change['new'][0]
        
        # Convert the 'content' bytes into a PIL Image
        input_image = Image.open(io.BytesIO(file_info['content']))
        
        # Process: Convert to Greyscale
        image_grey = input_image.convert('L')
        
        # Display the result
        print(f"Showing: {file_info['name']}")
        display(image_grey)

# 2. Set the observer to watch for the 'value' change
uploader.observe(on_upload_change, names='value')

# 3. Display the button and the output area
display(uploader, out)

In [ ]:
image_grey_np = np.array(image_grey)
print(f"Image shape: {image_grey_np.shape}")
print("Pixels: ")
print(image_grey_np)

In [ ]:
import numpy as np
import ipywidgets as widgets
from PIL import Image
from IPython.display import display, clear_output
import io


# Upload
uploader = widgets.FileUpload(accept='image/*', multiple=False)
output = widgets.Output()

image_grey = None
precomputed = {}


# Upload handler
def on_upload_change(change):
    global image_grey, precomputed

    precomputed = {}  # reset cache on new image

    with output:
        clear_output()

        if not change['new']:
            return

        file_info = change['new'][0]
        img = Image.open(io.BytesIO(file_info['content']))
        image_grey = img.convert('L')

        print("Uploaded:", file_info['name'])
        display(image_grey)

uploader.observe(on_upload_change, names='value')



# Convolution (fixed size)
def convolve2d_same(image, kernel):
    h, w = image.shape
    kh, kw = kernel.shape

    pad_h, pad_w = kh // 2, kw // 2

    padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='edge')

    output = np.zeros((h, w), dtype=np.float32)

    for i in range(h):
        for j in range(w):
            output[i, j] = np.sum(padded[i:i+kh, j:j+kw] * kernel)

    return np.clip(output, 0, 255)



# Kernels
blur_kernel = np.array([
    [1/9, 1/9, 1/9],
    [1/9, 1/9, 1/9],
    [1/9, 1/9, 1/9]
])

sharpen_kernel = np.array([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
])

extra_blur_kernel = extra_blur_kernel = np.array([
    [1, 4, 6, 4, 1],
    [4,16,24,16, 4],
    [6,24,36,24, 6],
    [4,16,24,16, 4],
    [1, 4, 6, 4, 1]
]) / 256



# Precompute
def precompute():
    global precomputed

    if image_grey is None:
        return

    img = np.array(image_grey)

    precomputed['blur'] = convolve2d_same(img, blur_kernel)
    precomputed['sharpen'] = convolve2d_same(img, sharpen_kernel)
    precomputed['extra_blur'] = convolve2d_same(img, extra_blur_kernel)



# Button action helper
def show_result(key, title):
    if image_grey is None:
        with output:
            clear_output()
            print("Upload image first!")
        return

    if key not in precomputed:
        precompute()

    result = Image.fromarray(precomputed[key].astype(np.uint8))

    with output:
        clear_output()
        print(title)
        display(result)



# Buttons
blur_btn = widgets.Button(description="Blur")
sharpen_btn = widgets.Button(description="Sharpen")
extra_blur_btn = widgets.Button(description="Extra Blur")



blur_btn.on_click(lambda b: show_result('blur', "Blurred Image"))
sharpen_btn.on_click(lambda b: show_result('sharpen', "Sharpened Image"))
extra_blur_btn.on_click(lambda b: show_result('extra_blur', "Extra Blurred Image"))



# UI
display(uploader,widgets.HBox([extra_blur_btn, blur_btn, sharpen_btn]),output)